# 02 — BERT Fake Review Detection

Fine-tunes **`bert-base-uncased`** for binary fake/genuine classification (project plan, Review-2: "BERT fine-tuning → evaluation").

- Input: review summary + review text
- Class imbalance handled with **class weights** in the cross-entropy loss (same as the CNN module)
- Transformers are trained on a **stratified subsample** of the training set (the paper benchmarks transformers on a 15,000-review subsample, Table 2); the evaluation uses the **full test set** so the results can be compared directly with the CNN
- BERT: bidirectional masked-language-model encoder (Devlin et al., 2019)

Runtime: **GPU (T4)**. Outputs go to `outputs/` in Drive.

In [ ]:
# --- Setup: mount Google Drive (dataset lives there, not in GitHub) ---
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/FakeReviewProject'   # change if your folder differs
DATA_DIR    = f'{PROJECT_DIR}/dataset'                     # train.csv, test.csv
OUT_DIR     = f'{PROJECT_DIR}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print(os.listdir(DATA_DIR))

In [ ]:
!pip -q install -U transformers accelerate sentencepiece
import torch, transformers, time
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
SEED = 42
transformers.set_seed(SEED)

In [ ]:
import pandas as pd, numpy as np, re

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

def prepare(df):
    # Module 1/2 cleaning: drop rows without text/label, fill missing summary
    df = df.dropna(subset=['review_text', 'label']).copy()
    df['review_summary'] = df['review_summary'].fillna('').astype(str)
    df['review_text'] = df['review_text'].astype(str)
    df['label'] = df['label'].astype(int)
    return df.reset_index(drop=True)

train_df, test_df = prepare(train_df), prepare(test_df)
print(train_df.shape, test_df.shape)
print(train_df.label.value_counts(), test_df.label.value_counts())

In [ ]:
# ---- Config ----
MODEL_NAME   = 'bert-base-uncased'
TRAIN_SAMPLE = 30000     # stratified subsample of train (set None for the full 291k; ~10x slower)
MAX_LEN      = 256
EPOCHS       = 2
BATCH        = 32
LR           = 2e-5

In [ ]:
from sklearn.model_selection import train_test_split
def text_of(df): return (df.review_summary + ' . ' + df.review_text).tolist()

sub = train_df if TRAIN_SAMPLE is None else train_df.groupby('label', group_keys=False).sample(
    frac=TRAIN_SAMPLE / len(train_df), random_state=SEED)
tr_df, va_df = train_test_split(sub, test_size=0.1, stratify=sub.label, random_state=SEED)
print(len(tr_df), len(va_df), tr_df.label.mean())

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDS(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tokenizer(text_of(df), truncation=True, max_length=MAX_LEN)
        self.y = df.label.values
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}; item['labels'] = int(self.y[i]); return item

ds_tr, ds_va, ds_te = ReviewDS(tr_df), ReviewDS(va_df), ReviewDS(test_df)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
cw = torch.tensor(compute_class_weight('balanced', classes=np.array([0, 1]), y=tr_df.label.values), dtype=torch.float)
print('class weights:', cw)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=cw.to(out.logits.device))
        return (loss, out) if return_outputs else loss

from sklearn.metrics import f1_score, roc_auc_score
def compute_metrics(p):
    prob = torch.softmax(torch.tensor(p.predictions), -1)[:, 1].numpy()
    return {'f1_fake': f1_score(p.label_ids, prob >= 0.5), 'roc_auc': roc_auc_score(p.label_ids, prob)}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
warmup = int(0.1 * EPOCHS * (len(ds_tr) // BATCH))   # 10% linear warmup
args = TrainingArguments(
    output_dir=f'/content/{MODEL_NAME.replace("/", "_")}', num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 4,
    learning_rate=LR, weight_decay=0.01, warmup_steps=warmup, fp16=torch.cuda.is_available(),
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='f1_fake', save_total_limit=1, logging_steps=100, report_to='none', seed=SEED)
trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                          data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)

In [ ]:
t0 = time.time()
trainer.train()
train_time = time.time() - t0
print(f'Training time: {train_time:.1f}s')

## Evaluation on the full test set

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

def evaluate(name, y_true, y_prob, categories, threshold=0.5, extra=None):
    y_pred = (y_prob >= threshold).astype(int)
    m = {
        'model': name,
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),   # fake class
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1_fake':   f1_score(y_true, y_pred, zero_division=0),
        'f1_macro':  f1_score(y_true, y_pred, average='macro'),
        'roc_auc':   roc_auc_score(y_true, y_prob),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'n_test': int(len(y_true)),
    }
    if extra: m.update(extra)
    # per-category results (paper Table 6)
    per_cat = {}
    for c in sorted(pd.unique(categories)):
        idx = np.asarray(categories) == c
        yt, yp, pr = y_true[idx], y_pred[idx], y_prob[idx]
        per_cat[c] = {
            'roc_auc':   roc_auc_score(yt, pr) if len(np.unique(yt)) > 1 else None,
            'precision': precision_score(yt, yp, zero_division=0),
            'recall':    recall_score(yt, yp, zero_division=0),
            'f1_fake':   f1_score(yt, yp, zero_division=0),
        }
    m['per_category'] = per_cat
    print(classification_report(y_true, y_pred, target_names=['Genuine', 'Fake'], digits=4))
    print(pd.DataFrame(per_cat).T.round(4))
    key = name.lower()
    with open(f'{OUT_DIR}/{key}_metrics.json', 'w') as f:
        json.dump(m, f, indent=2)
    np.save(f'{OUT_DIR}/{key}_test_probs.npy', y_prob)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items() if k != 'per_category'})
    return m

In [ ]:
val_prob = torch.softmax(torch.tensor(trainer.predict(ds_va).predictions), -1)[:, 1].numpy()
ths = np.arange(0.05, 0.96, 0.01)
best_t = ths[np.argmax([f1_score(va_df.label.values, val_prob >= t) for t in ths])]
print('Validation-selected threshold:', round(best_t, 2))

t0 = time.time()
test_prob = torch.softmax(torch.tensor(trainer.predict(ds_te).predictions), -1)[:, 1].numpy()
m = evaluate('BERT', test_df.label.values, test_prob, test_df.category.values, threshold=best_t,
             extra={'threshold': float(best_t), 'train_time_s': train_time,
                    'infer_time_s': time.time() - t0, 'train_size': int(len(tr_df))})

In [ ]:
# Save the fine-tuned model to Drive
save_dir = f'{OUT_DIR}/bert_model'
trainer.save_model(save_dir); tokenizer.save_pretrained(save_dir)
with open(f'{save_dir}/predict_config.json', 'w') as f:
    json.dump({'max_len': MAX_LEN, 'threshold': float(best_t), 'base_model': MODEL_NAME}, f, indent=2)

def predict_review(text, summary=''):
    enc = tokenizer(summary + ' . ' + text, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(model.device)
    with torch.no_grad(): p = torch.softmax(model(**enc).logits, -1)[0, 1].item()
    return ('FAKE' if p >= best_t else 'GENUINE'), round(p, 4)

predict_review('Absolutely amazing product!!! Best purchase ever, everyone should buy this, five stars!!!', 'Amazing')